# 06. Building Your First Complete Agent

Welcome to the capstone of the beginner curriculum. We are going to build **one complete, bounded, testable agent** using the hybrid architecture typical of real enterprise systems.

**Scenario:** A customer escalated ticket T-102: "I was charged twice for my subscription. Support has not resolved this. Please fix it."

This implementation demonstrates professional separation of domain models, policy, runtime orchestration, a model adapter, and secure side-effect execution.

## Part 1: Define the Domain Fixtures
First, we create deterministic fixtures simulating a real business database.

In [1]:
import json
import logging
from typing import Optional, Literal, Any
from pydantic import BaseModel, Field

logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger('Northstar')

DB = {
    'tickets': {'T-102': {'customer_id': 'C-55', 'text': 'I was charged twice for my subscription. Please fix it.', 'status': 'open'}},
    'transactions': {'C-55': [
        {'tx_id': 'TX-901', 'amount_cents': 10000, 'date': '2026-08-01', 'note': 'initial charge'},
        {'tx_id': 'TX-902', 'amount_cents': 10000, 'date': '2026-08-01', 'note': 'system duplicate'}
    ]},
    'refunds': []
}

print('Domain fixtures initialized.')

Domain fixtures initialized.


## Part 2: Define Trusted Context
The LLM does not invent its own identity. We define an application-provided trusted context containing roles and tenant identity.

In [2]:
class ExecutionContext(BaseModel):
    user_id: str
    tenant_id: str
    roles: set[str] = Field(default_factory=set)
    request_id: str

ctx = ExecutionContext(user_id='U-88', tenant_id='Northstar', roles={'support:read', 'billing:read', 'refund:issue'}, request_id='REQ-999')
print(f'Running as: {ctx}')

Running as: user_id='U-88' tenant_id='Northstar' roles={'billing:read', 'support:read', 'refund:issue'} request_id='REQ-999'


## Part 3: Typed Tool I/O Models
Tools must be strictly typed for both inputs and outputs. We avoid raw JSON strings.

In [3]:
# Output Models
class TicketDetails(BaseModel):
    customer_id: str
    text: str
    status: str

class Transaction(BaseModel):
    tx_id: str
    amount_cents: int
    date: str
    note: Optional[str] = None

class TransactionList(BaseModel):
    transactions: list[Transaction] = Field(default_factory=list)

class RefundPolicy(BaseModel):
    policy_text: str

# Input Models
class GetTicketArgs(BaseModel):
    ticket_id: str

class GetCustomerArgs(BaseModel):
    customer_id: str

class GetRefundPolicyArgs(BaseModel):
    pass

def get_ticket_details(args: GetTicketArgs) -> TicketDetails:
    if args.ticket_id not in DB['tickets']:
        raise ValueError('Ticket not found')
    return TicketDetails(**DB['tickets'][args.ticket_id])

def get_recent_transactions(args: GetCustomerArgs) -> TransactionList:
    if args.customer_id not in DB['transactions']:
        raise ValueError('No transactions found')
    return TransactionList(transactions=[Transaction(**t) for t in DB['transactions'][args.customer_id]])

def get_refund_policy(args: GetRefundPolicyArgs) -> RefundPolicy:
    return RefundPolicy(policy_text='Duplicate charges must be refunded in full. Refunds require human manager approval.')


## Part 4: Proposals & Approvals
When the agent decides an action is needed, it produces a `RefundProposal`. A human manager provides an `Approval` bound to a cryptographic digest of the proposal.

In [4]:
import hashlib

class RefundProposal(BaseModel):
    customer_id: str
    transaction_id: str
    amount_cents: int
    reason: str

class Approval(BaseModel):
    proposal_digest: str
    approver_id: str
    decision: Literal['approve', 'reject']

def hash_proposal(proposal: RefundProposal) -> str:
    return hashlib.sha256(proposal.model_dump_json().encode()).hexdigest()


## Part 5: The Consequential Write Tool (Refund)
This function is protected and requires an idempotency key. We will enforce that it can ONLY be executed by the Dispatcher.

In [5]:
class IssueRefundArgs(BaseModel):
    customer_id: str
    transaction_id: str
    amount_cents: int
    idempotency_key: str

class RefundResult(BaseModel):
    status: Literal['success', 'already_processed']
    amount_cents: int
    transaction_id: str

def _issue_refund_impl(args: IssueRefundArgs) -> RefundResult:
    # 1. Idempotency Check
    for r in DB['refunds']:
        if r['idempotency_key'] == args.idempotency_key:
            return RefundResult(status='already_processed', amount_cents=args.amount_cents, transaction_id=args.transaction_id)
    
    # 2. Execution
    DB['refunds'].append(args.model_dump())
    return RefundResult(status='success', amount_cents=args.amount_cents, transaction_id=args.transaction_id)


## Part 6: Tool Registry & Secure Dispatcher
We define `ToolDefinition` explicitly (avoiding Pydantic shadowing warnings). Our `dispatch_tool` centralizes validation, authorization, and business rules.

In [6]:
from typing import Callable, Union

class ToolDefinition(BaseModel):
    name: str
    effect: Literal['READ_ONLY', 'CONSEQUENTIAL_WRITE']
    required_permission: str
    input_model: type[BaseModel]
    result_model: type[BaseModel]
    func: Callable

TOOL_REGISTRY = {
    'get_ticket_details': ToolDefinition(name='get_ticket_details', effect='READ_ONLY', required_permission='support:read', input_model=GetTicketArgs, result_model=TicketDetails, func=get_ticket_details),
    'get_recent_transactions': ToolDefinition(name='get_recent_transactions', effect='READ_ONLY', required_permission='billing:read', input_model=GetCustomerArgs, result_model=TransactionList, func=get_recent_transactions),
    'get_refund_policy': ToolDefinition(name='get_refund_policy', effect='READ_ONLY', required_permission='support:read', input_model=GetRefundPolicyArgs, result_model=RefundPolicy, func=get_refund_policy),
    'issue_refund': ToolDefinition(name='issue_refund', effect='CONSEQUENTIAL_WRITE', required_permission='refund:issue', input_model=IssueRefundArgs, result_model=RefundResult, func=_issue_refund_impl),
}

class ToolError(BaseModel):
    error: str

def dispatch_tool(tool_name: str, args_dict: dict, ctx: ExecutionContext) -> Union[BaseModel, ToolError]:
    if tool_name not in TOOL_REGISTRY:
        return ToolError(error=f'Unknown tool: {tool_name}')
        
    tdef = TOOL_REGISTRY[tool_name]
    
    if tdef.required_permission not in ctx.roles:
        return ToolError(error=f'Authorization denied. Missing role: {tdef.required_permission}')
        
    try:
        args_obj = tdef.input_model(**args_dict)
    except Exception as e:
        return ToolError(error=f'Schema validation error: {str(e)}')

    # BUSINESS VALIDATION FOR WRITES
    if tool_name == 'issue_refund':
        # Verify transaction belongs to customer
        customer_txs = DB['transactions'].get(args_obj.customer_id, [])
        tx = next((t for t in customer_txs if t['tx_id'] == args_obj.transaction_id), None)
        if not tx:
            return ToolError(error='Transaction not found for this customer.')
        if tx['amount_cents'] != args_obj.amount_cents:
            return ToolError(error='Refund amount must match transaction amount strictly.')
        # Verify it is a duplicate (simplified heuristic for demo)
        if 'duplicate' not in tx.get('note', '').lower():
            return ToolError(error='Transaction is not marked as a duplicate charge.')
            
    try:
        result = tdef.func(args_obj)
        return result
    except Exception as e:
        return ToolError(error=str(e))


## Part 7: Runtime Models & State
We define strict terminal reasons and fix mutable defaults.

In [7]:
class ToolCall(BaseModel):
    id: str
    name: str
    arguments: dict

class AgentDecision(BaseModel):
    tool_calls: list[ToolCall] = Field(default_factory=list)
    proposal: Optional[RefundProposal] = None
    final_answer: Optional[str] = None

class AgentState(BaseModel):
    ticket_id: str
    steps: int = 0
    max_steps: int = 6
    history: list[dict] = Field(default_factory=list)
    seen_actions: set[str] = Field(default_factory=set)
    terminal_reason: Optional[Literal['SUCCESS', 'NO_REFUND_NEEDED', 'INSUFFICIENT_EVIDENCE', 'AUTHORIZATION_DENIED', 'APPROVAL_REQUIRED', 'HUMAN_REJECTED', 'STEP_BUDGET_EXHAUSTED', 'NO_PROGRESS']] = None
    proposal: Optional[RefundProposal] = None
    approval: Optional[Approval] = None


## Part 8: The Evidence-Based Mock Model
Instead of returning fixed results by turn, this mock looks at the `history` (the gathered evidence) to make decisions, teaching true stateful decision logic.

In [8]:
class MockDecisionModel:
    def decide(self, state: AgentState) -> AgentDecision:
        history_str = json.dumps(state.history)
        
        # 1. Need ticket details?
        if not any(h['tool'] == 'get_ticket_details' for h in state.history):
            return AgentDecision(tool_calls=[ToolCall(id='t1', name='get_ticket_details', arguments={'ticket_id': state.ticket_id})])
            
        # 2. Need customer transactions & policy?
        if not any(h['tool'] == 'get_recent_transactions' for h in state.history):
            return AgentDecision(tool_calls=[
                ToolCall(id='t2', name='get_recent_transactions', arguments={'customer_id': 'C-55'}),
                ToolCall(id='t3', name='get_refund_policy', arguments={})
            ])
            
        # 3. Analyze evidence
        if 'charged twice' in history_str and 'TX-902' in history_str:
            return AgentDecision(proposal=RefundProposal(
                customer_id='C-55', transaction_id='TX-902', amount_cents=10000, reason='Duplicate charge identified in TX-902.'
            ))
            
        return AgentDecision(final_answer='I investigated but could not find a duplicate charge.')


## Part 9: The Core Execution Loop
Handles autonomous looping, human injection, budgets, and idempotent execution.

In [9]:
def run_agent(ctx: ExecutionContext, state: AgentState, model) -> AgentState:
    # If resuming with an approval, process it immediately.
    if state.proposal and state.approval:
        if state.approval.decision == 'reject':
            state.terminal_reason = 'HUMAN_REJECTED'
            return state
            
        check_digest = hash_proposal(state.proposal)
        if check_digest != state.approval.proposal_digest:
            state.terminal_reason = 'AUTHORIZATION_DENIED'
            return state
            
        idem_key = f'ref_{state.proposal.transaction_id}_{check_digest[:8]}'
        args = {'customer_id': state.proposal.customer_id, 'transaction_id': state.proposal.transaction_id, 'amount_cents': state.proposal.amount_cents, 'idempotency_key': idem_key}
        
        # Dispatch consequential write
        res = dispatch_tool('issue_refund', args, ctx)
        state.history.append({'tool': 'issue_refund', 'result': res.model_dump()})
        
        if isinstance(res, RefundResult):
            # POSTCONDITION CHECK
            found = any(r['idempotency_key'] == idem_key for r in DB['refunds'])
            if found:
                state.terminal_reason = 'SUCCESS'
            else:
                state.terminal_reason = 'INSUFFICIENT_EVIDENCE'
        else:
            state.terminal_reason = 'AUTHORIZATION_DENIED'
        return state

    while state.terminal_reason is None:
        if state.steps >= state.max_steps:
            state.terminal_reason = 'STEP_BUDGET_EXHAUSTED'
            break
            
        state.steps += 1
        decision = model.decide(state)
        
        if decision.final_answer:
            state.terminal_reason = 'NO_REFUND_NEEDED'
            break
            
        if decision.proposal:
            state.proposal = decision.proposal
            state.terminal_reason = 'APPROVAL_REQUIRED'
            break
            
        for tc in decision.tool_calls:
            fingerprint = f'{tc.name}:{json.dumps(tc.arguments, sort_keys=True)}'
            if fingerprint in state.seen_actions:
                state.terminal_reason = 'NO_PROGRESS'
                break
            state.seen_actions.add(fingerprint)
            
            if TOOL_REGISTRY[tc.name].effect != 'READ_ONLY':
                state.terminal_reason = 'AUTHORIZATION_DENIED'
                break
                
            res = dispatch_tool(tc.name, tc.arguments, ctx)
            state.history.append({'tool': tc.name, 'result': res.model_dump()})
            
    return state


## Part 10: Exhaustive Test Suite & Invariants
We assert that invariants hold true across various scenarios.

In [10]:
print('--- Test 1: Happy Path ---')
state1 = AgentState(ticket_id='T-102')
model = MockDecisionModel()
state1 = run_agent(ctx, state1, model)
assert state1.terminal_reason == 'APPROVAL_REQUIRED'

digest = hash_proposal(state1.proposal)
state1.approval = Approval(proposal_digest=digest, approver_id='MGR-1', decision='approve')
state1 = run_agent(ctx, state1, model)
assert state1.terminal_reason == 'SUCCESS'
print('Happy path passed.')

print('--- Test 2: Idempotency (Duplicate Execution) ---')
state1.terminal_reason = None # reset to test retry
state1 = run_agent(ctx, state1, model)
assert state1.terminal_reason == 'SUCCESS'
assert len(DB['refunds']) == 1 # Second execution did not duplicate!
print('Idempotency passed.')

print('--- Test 3: No Progress Detection ---')
class LoopModel:
    def decide(self, state): return AgentDecision(tool_calls=[ToolCall(id='1', name='get_ticket_details', arguments={'ticket_id': 'T-102'})])
state3 = run_agent(ctx, AgentState(ticket_id='T-102'), LoopModel())
assert state3.terminal_reason == 'NO_PROGRESS'
print('No-progress detection passed.')

print('--- Test 4: Business Validation (Wrong Amount) ---')
state4 = AgentState(ticket_id='T-102', proposal=RefundProposal(customer_id='C-55', transaction_id='TX-902', amount_cents=999999, reason='bad amount'))
state4.approval = Approval(proposal_digest=hash_proposal(state4.proposal), approver_id='M-1', decision='approve')
state4 = run_agent(ctx, state4, model)
assert state4.terminal_reason == 'AUTHORIZATION_DENIED'
print('Business validation passed.')


--- Test 1: Happy Path ---
Happy path passed.
--- Test 2: Idempotency (Duplicate Execution) ---
Idempotency passed.
--- Test 3: No Progress Detection ---
No-progress detection passed.
--- Test 4: Business Validation (Wrong Amount) ---
Business validation passed.


## Part 11: Optional Real OpenAI Implementation
If you have an `OPENAI_API_KEY`, we run the *exact same loop* using a real LLM. Note that we DO NOT give the LLM the `issue_refund` tool; we give it a `propose_refund` tool, ensuring it generates a `RefundProposal` instead of a direct side-effect.

In [11]:
import os
if os.getenv('OPENAI_API_KEY'):
    from openai import OpenAI
    class OpenAIDecisionModel:
        def __init__(self):
            self.client = OpenAI()
            self.messages = [{'role': 'system', 'content': 'You are a support agent. Investigate duplicate charges. If found, use propose_refund. Otherwise output a final answer.'}]
            self.tools = [
                {'type': 'function', 'function': {'name': 'get_ticket_details', 'parameters': GetTicketArgs.model_json_schema()}},
                {'type': 'function', 'function': {'name': 'get_recent_transactions', 'parameters': GetCustomerArgs.model_json_schema()}},
                {'type': 'function', 'function': {'name': 'propose_refund', 'parameters': RefundProposal.model_json_schema()}}
            ]
        def decide(self, state: AgentState) -> AgentDecision:
            if len(self.messages) == 1:
                self.messages.append({'role': 'user', 'content': f'Investigate ticket {state.ticket_id}'})
            for h in state.history:
                self.messages.append({'role': 'system', 'content': f'Tool Result: {h}'})
            state.history.clear()
            resp = self.client.chat.completions.create(model='gpt-4o-mini', messages=self.messages, tools=self.tools)
            msg = resp.choices[0].message
            self.messages.append(msg)
            if not msg.tool_calls:
                return AgentDecision(final_answer=msg.content)
            tcs = []
            for tc in msg.tool_calls:
                if tc.function.name == 'propose_refund':
                    return AgentDecision(proposal=RefundProposal(**json.loads(tc.function.arguments)))
                tcs.append(ToolCall(id=tc.id, name=tc.function.name, arguments=json.loads(tc.function.arguments)))
            return AgentDecision(tool_calls=tcs)
    print('Running Real OpenAI Agent...')
    real_state = run_agent(ctx, AgentState(ticket_id='T-102'), OpenAIDecisionModel())
    print(f'LLM reached terminal state: {real_state.terminal_reason}')
    if real_state.proposal:
        print('LLM Proposal:', real_state.proposal)
else:
    print('Skipping real execution. Set OPENAI_API_KEY to run.')


Skipping real execution. Set OPENAI_API_KEY to run.
